# SpaceX Falcon 9 - Interactive Visual Analytics with Folium
IBM Applied Data Science Capstone

In [ ]:
!pip install folium -q
import folium
import pandas as pd
import wget
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

In [ ]:
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = pd.read_csv(spacex_csv_file)
spacex_df.head()

In [ ]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

## Mark all launch sites on a map

In [ ]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=4)

for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)
site_map

## Mark success/failed launches

In [ ]:
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

for index, record in spacex_df.iterrows():
    folium.Marker(location=[record['Lat'], record['Long']],
                  icon=folium.Icon(color='white', icon_color=record['marker_color'])).add_to(marker_cluster)
site_map

## Calculate distances to coastline / railway / highway (example)

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Example coastline coordinate near CCAFS
coastline_lat, coastline_lon = 28.56367, -80.57163
launch_lat, launch_lon = 28.563197, -80.576820
distance_coastline = calculate_distance(launch_lat, launch_lon, coastline_lat, coastline_lon)
print(f'Distance to coastline: {distance_coastline:.2f} km')